In [ ]:
import pandas as pd
import numpy as np
from chembl_webresource_client.new_client import new_client

activity = new_client.activity

# res = activity.filter(
#     target_chembl_id="CHEMBL203",  # EGFR
#     molecule_chembl_id={"in": ["CHEMBL1079742", "CHEMBL941", "CHEMBL2105758"]}, # erlotinib, gefitinib, osimertinib
#     standard_type="IC50",
# )

ligand = activity.filter(
    target_chembl_id="CHEMBL203",  # EGFR
    molecule_chembl_id__in=[
        "CHEMBL1079742",  # erlotinib
        "CHEMBL941",      # gefitinib
        "CHEMBL2105758"   # osimertinib
    ]
)

ligand_df = pd.DataFrame(ligand)

# api data cleaning and keep only IC50 and Kd for EGFR
ligand_df = ligand_df[ligand_df["standard_type"].isin(["IC50", "Kd"])]
ligand_df = ligand_df[ligand_df["standard_units"] == "nM"]
ligand_df = ligand_df.dropna(subset=["standard_value"])

# convert to numeric
ligand_df["standard_value"] = pd.to_numeric(ligand_df["standard_value"], errors="coerce")
ligand_df = ligand_df.dropna(subset=["standard_value"])
ligand_df["value_M"] = ligand_df["standard_value"] * 1e-9

# log transform
ligand_df["p_value"] = -np.log10(ligand_df["value_M"])
ligand_df["binding_type"] = ligand_df["standard_type"]

# rename for clarity
ligand_df = ligand_df.rename(columns={
    "molecule_chembl_id": "ligand",
    "target_chembl_id": "protein",
    "p_value": "p_binding"
})

final_ligand_df = ligand_df[[
    "ligand",
    "protein",
    "binding_type",
    "standard_value",
    "standard_units",
    "p_binding"
]]

print(final_ligand_df.head())
print("\nSummary:")
print(final_ligand_df["binding_type"].value_counts())
final_ligand_df.to_csv("data/egfr_ligand_binding.csv", index=False)

      ligand    protein binding_type  standard_value standard_units  p_binding
0  CHEMBL941  CHEMBL203         IC50        100000.0             nM        4.0
2  CHEMBL941  CHEMBL203           Kd         10000.0             nM        5.0
3  CHEMBL941  CHEMBL203           Kd         10000.0             nM        5.0
4  CHEMBL941  CHEMBL203           Kd         10000.0             nM        5.0
5  CHEMBL941  CHEMBL203           Kd         10000.0             nM        5.0

Summary:
binding_type
Kd      23
IC50     6
Name: count, dtype: int64


In [17]:
cptac_model_df = pd.read_csv("data/cptac_model_df.csv")
cptac_model_df["key"] = 1
final_ligand_df["key"] = 1

model_df = cptac_model_df.merge(final_ligand_df, on="key").drop("key", axis=1)
model_df.to_csv("data/model_df.csv", index=False)
#print(model_df.columns) #\_{!!}_/

#need patient mutation column
#ligand structure SMILES

mutation = pd.read_csv("data/mutation_features_cleaned.csv")

import re
def label_egfr_hotspot(mutation_value: str) -> str:
    m = str(mutation_value).upper().strip()

    if (
        "EXON 19" in m
        or "19DEL" in m
        or "DEL19" in m
        or re.search(r"E\d+_A\d+DEL", m)
        or re.search(r"L\d+_A\d+DEL", m)
        or re.search(r"L\d+_T\d+DEL", m)
        or "DELINS" in m
    ):
        return "exon19del"
    elif "L858R" in m:
        return "L858R"
    elif "T790M" in m:
        return "T790M"
    elif "C797S" in m:
        return "C797S"
    elif "G719" in m:
        return "G719X"
    elif "L861Q" in m:
        return "L861Q"
    elif "S768I" in m:
        return "S768I"
    elif "EXON 20" in m or "INS" in m or "DUP" in m:
        return "exon20_alteration"
    else:
        return "other"

mutation["egfr_hotspot_label"] = mutation["mutation"].apply(label_egfr_hotspot)
mutation["is_egfr_hotspot"] = mutation["egfr_hotspot_label"].ne("other").astype(int)

patient_mut = mutation[["patient_id", "mutation", "egfr_hotspot_label"]].drop_duplicates()
patient_mut.to_csv("data/patient_mutation_labels.csv", index=False)


C:\Users\anaso\AppData\Local\Temp\ipykernel_12268\3095247271.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  final_ligand_df["key"] = 1
